In [1]:
from QLBM import QLBMV2, collision, InitializeQC
import numpy as np
import matplotlib.pyplot as plt
import qiskit_aer
from qiskit import transpile

In [2]:
# Domain and grid setup
N_POINTS_X, N_POINTS_Y, N_POINTS_Z = 32, 32, 32
# Simulation parameters

NUMBER_DISCRETE_VELOCITIES = 27  # D2Q9 lattice configuration

TIMESTEPS = 5000

In [ ]:
Qq = 27
Nx = N_POINTS_X-1
Ny = N_POINTS_Y-1
Nz = N_POINTS_Y-1
dx = 1
dy = 1
dz = 1
dt = dx
c = dt/dx
Re = 100
f_eq = np.zeros((Qq, Nx+1, Ny+1, Ny+1))
f = np.zeros((Qq, Nx+1, Ny+1, Ny+1))
f_star = np.zeros((Qq, Nx+1, Ny+1, Ny+1))
Lx = dx * float(Nx+1)
Ly = dy * float(Ny+1)
Lz = dz * float(Nz+1)
U = 0.1
nu = U*Lx/Re
nu_star = 1/6*dt
cs = np.sqrt(c**2/3)
tau_f = 1.0
rho_0 = 1.0
rho = np.zeros((Nx+1, Ny+1, Ny+1))
u = np.zeros((Nx+1, Ny+1, Ny+1, 3))
e = np.array([
    [0,  1, -1,  0,  0,   0,  0,  1, -1,  1, -1,  0,  0,  1, -1,  1, -1,  0,  0,  1, -1,  1, -1,  1, -1, -1,  1],
    [0,  0,  0,  1, -1,   0,  0,  1, -1,  0,  0,  1, -1, -1,  1,  0,  0,  1, -1,  1, -1,  1, -1, -1,  1,  1, -1],
    [0,  0,  0,  0,  0,   1, -1,  0,  0,  1, -1,  1, -1,  0,  0, -1,  1, -1,  1,  1, -1, -1,  1,  1, -1,  1, -1]
]).T
w = np.array([
    8/27,                   
    2/27, 2/27, 2/27, 2/27, 2/27, 2/27,  
    1/54, 1/54, 1/54, 1/54, 1/54, 1/54, 1/54, 1/54, 1/54, 1/54, 1/54, 1/54,  
    1/216, 1/216, 1/216, 1/216, 1/216, 1/216, 1/216, 1/216  
])

u_n = u[:, :, :, 0].copy()
v_n = u[:, :, :, 1].copy() 
w_n = u[:, :, :, 2].copy()
u_t = np.zeros((Nx+3, Ny+3, Ny+3))
v_t = np.zeros((Nx+3, Ny+3, Ny+3))
w_t = np.zeros((Nx+3, Ny+3, Ny+3))

uq_t = np.zeros((Nx+3, Ny+3, Ny+3))
vq_t = np.zeros((Nx+3, Ny+3, Ny+3))
wq_t = np.zeros((Nx+3, Ny+3, Ny+3))

u_d = np.zeros((Nx+1, Ny+1, Ny+1))
v_d = np.zeros((Nx+1, Ny+1, Ny+1))
w_d = np.zeros((Nx+1, Ny+1, Ny+1))
print(nu,nu_star,Lx,Ly,TIMESTEPS)

q_error=[]
c_error=[]
qc_error=[]

(27, 3)
0.032 0.16666666666666666 32.0 32.0 5000


In [ ]:
##Initial
x0 = np.linspace(0,Lx,Nx+1)
y0 = np.linspace(0,Ly,Ny+1)
z0 = np.linspace(0,Lz,Nz+1)

X0,Y0,Z0 = np.meshgrid(x0,y0,z0)
rho[:, :, :] = rho_0
u[:, :, Nz, 0] = U
u[:, :, :, 1] = 0
u[:, :, :, 2] = 0

def possion_uu(u_hat,delta_x):
    vor_u = np.zeros((u_hat.shape[0]+2,u_hat.shape[0]+2,u_hat.shape[0]+2))

    vor_u[1:-1,1:-1,1:-1] = u_hat
    vor_u[:,:, 0] = 4.0*vor_u[:,:,1]-6.0*vor_u[:,:,2] + 4.0*vor_u[:,:,3] - vor_u[:,:,4]
    vor_u[:,:, -1] = 4.0*vor_u[:,:,-2]-6.0*vor_u[:,:,-3] + 4.0*vor_u[:,:,-4] - vor_u[:,:,-5]
    vor_u[0, :,:] = 4.0*vor_u[1, :,:] - 6.0*vor_u[2, :,:] + 4.0*vor_u[3, :,:] - vor_u[4, :,:]
    vor_u[-1, :,:] = 4.0*vor_u[-2,:,:] - 6.0*vor_u[-3,:,:] + 4.0*vor_u[-4,:,:] - vor_u[-5,:,:]
    vor_u[:, 0,:] = 4.0*vor_u[:,1,:] - 6.0*vor_u[:,2,:] + 4.0*vor_u[:,3,:] - vor_u[:,4,:]
    vor_u[:, -1,:] = 4.0*vor_u[:,-2,:] - 6.0*vor_u[:,-3,:] + 4.0*vor_u[:,-4,:] - vor_u[:,-5,:]
    """vor_u1 = (-28*vor_u[2:-2,2:-2]-15*(vor_u[1:-3,2:-2] + vor_u[3:-1,2:-2] + vor_u[2:-2,1:-3] + vor_u[2:-2,3:-1]))/(77*delta_x**2) - \
            2.0*(vor_u[3:-1,3:-1] + vor_u[1:-3,3:-1] + vor_u[1:-3,1:-3] + vor_u[3:-1,1:-3])/(77*delta_x**2) + \
            12.0*(vor_u[0:-4,2:-2] + vor_u[4:,2:-2] + vor_u[2:-2,0:-4] + vor_u[2:-2,4:])/(77*delta_x**2)"""
    vor_u1 = 0.5*((-4*vor_u[1:-1,1:-1,1:-1]-(vor_u[0:-2,1:-1,1:-1] + vor_u[2:,1:-1,1:-1] + vor_u[1:-1,0:-2,1:-1] + vor_u[1:-1,2:,1:-1]))/(3*delta_x**2) + \
            2.0*(vor_u[2:,2:,1:-1] + vor_u[0:-2,2:,1:-1] + vor_u[0:-2,0:-2,1:-1] + vor_u[2:,0:-2,1:-1])/(3*delta_x**2)+\
                (-4*vor_u[1:-1,1:-1,1:-1]-(vor_u[0:-2,1:-1,1:-1] + vor_u[2:,1:-1,1:-1] + vor_u[1:-1,1:-1,0:-2] + vor_u[1:-1,1:-1,2:]))/(3*delta_x**2) + \
            2.0*(vor_u[2:,1:-1,2:] + vor_u[0:-2,1:-1,2:] + vor_u[0:-2,1:-1,0:-2] + vor_u[2:,1:-1,0:-2])/(3*delta_x**2) +\
                (-4*vor_u[1:-1,1:-1,1:-1]-(vor_u[1:-1,0:-2,1:-1] + vor_u[1:-1,2:,1:-1] + vor_u[1:-1,1:-1,0:-2] + vor_u[1:-1,1:-1,2:]))/(3*delta_x**2) + \
            2.0*(vor_u[1:-1,2:,2:] + vor_u[1:-1,0:-2,2:] + vor_u[1:-1,0:-2,0:-2] + vor_u[1:-1,2:,0:-2])/(3*delta_x**2))
    """vor_u1 = (-20.0*vor_u[2:-2,2:-2]+4.0*(vor_u[1:-3,2:-2] + vor_u[3:-1,2:-2] + vor_u[2:-2,1:-3] + vor_u[2:-2,3:-1]))/(6*delta_x**2) + \
            1.0*(vor_u[3:-1,3:-1] + vor_u[1:-3,3:-1] + vor_u[1:-3,1:-3] + vor_u[3:-1,1:-3])/(6*delta_x**2)"""

    return vor_u1

[ 0.          1.03225806  2.06451613  3.09677419  4.12903226  5.16129032
  6.19354839  7.22580645  8.25806452  9.29032258 10.32258065 11.35483871
 12.38709677 13.41935484 14.4516129  15.48387097 16.51612903 17.5483871
 18.58064516 19.61290323 20.64516129 21.67741935 22.70967742 23.74193548
 24.77419355 25.80645161 26.83870968 27.87096774 28.90322581 29.93548387
 30.96774194 32.        ] [ 0.          1.03225806  2.06451613  3.09677419  4.12903226  5.16129032
  6.19354839  7.22580645  8.25806452  9.29032258 10.32258065 11.35483871
 12.38709677 13.41935484 14.4516129  15.48387097 16.51612903 17.5483871
 18.58064516 19.61290323 20.64516129 21.67741935 22.70967742 23.74193548
 24.77419355 25.80645161 26.83870968 27.87096774 28.90322581 29.93548387
 30.96774194 32.        ] [ 0.          1.03225806  2.06451613  3.09677419  4.12903226  5.16129032
  6.19354839  7.22580645  8.25806452  9.29032258 10.32258065 11.35483871
 12.38709677 13.41935484 14.4516129  15.48387097 16.51612903 17.5483871
 1

In [5]:
simulator = qiskit_aer.backends.statevector_simulator.StatevectorSimulator()

In [ ]:
# Initialize the quantum LBM scalar field
Psi_qlbm = np.zeros((TIMESTEPS + 1, N_POINTS_X, N_POINTS_Y, N_POINTS_Z))
Psi_qlbm1 = np.zeros((TIMESTEPS + 1, N_POINTS_X, N_POINTS_Y, N_POINTS_Z))
Psi_qlbm2 = np.zeros((TIMESTEPS + 1, N_POINTS_X, N_POINTS_Y, N_POINTS_Z))
Psi_qlbm3 = np.zeros((TIMESTEPS + 1, N_POINTS_X, N_POINTS_Y, N_POINTS_Z))
Psi_qlbm[0, :, :, :] = rho_0#Psi_init
Psi_qlbm1[0, :, :, :] = u[:,:,:,0].copy()#Psi_init
Psi_qlbm2[0, :, :, :] = u[:,:,:,1].copy()#Psi_init
Psi_qlbm3[0, :, :, :] = u[:,:,:,2].copy()#Psi_init
Psi_qlbm0 = Psi_qlbm[0,:,:,:].copy()
u_LBM = np.zeros((N_POINTS_X, N_POINTS_Y, N_POINTS_Z, 3))
u_LBM[:, :, :, 0] = Psi_qlbm1[0, :, :,:]  # Set the x-component of the velocity
u_LBM[:, :, :, 1] = Psi_qlbm2[0, :, :,:]  # Set the y-component of the velocity
u_LBM[:, :, :, 2] = Psi_qlbm3[0, :, :,:]  # Set the z-component of the velocity

# Quantum LBM simulation loop
for t in range(TIMESTEPS):
    u_d = possion_uu(u[:,:,:,0],dx)
    v_d = possion_uu(u[:,:,:,1],dx)
    w_d = possion_uu(u[:,:,:,2],dx)
    temp = u[:, :, :, 0] * u[:, :, :, 0] + u[:, :, :, 1] * u[:, :, :, 1]+ u[:, :, :, 2] * u[:, :, :, 2]
    for Q in range(27):
        f_eq[Q,:, :, :] = w[Q]*rho*(1.0 + 3.0 * (e[Q, 0] * u[:, :, :, 0] + e[Q, 1] * u[:, :, :, 1] + e[Q, 2] * u[:, :, :, 2]) + 4.5 * (e[Q, 0] * u[:, :, :, 0] + e[Q, 1] * u[:, :, :, 1] + e[Q, 2] * u[:, :, :, 2]) ** 2 - 1.5 * temp)
        f[Q, :, :, :] = f[Q, :, :, :] - (f[Q, :, :, :] - f_eq[Q, :, :, :]) / tau_f

    for vvv in range(27):
        f[vvv, :, :, :] = np.roll(
            np.roll(
                np.roll(f[vvv, :, :, :], e[vvv, 0], axis=0),
                e[vvv, 1], axis=1
            ),
            e[vvv, 2], axis=2
        )

    rho = np.sum(f, axis=0)
    u[:, :, :, 0] = np.sum(f * e[:,0].reshape(27,1,1,1), axis=0) / rho
    u[:, :, :, 1] = np.sum(f * e[:,1].reshape(27,1,1,1), axis=0) / rho
    u[:, :, :, 2] = np.sum(f * e[:,2].reshape(27,1,1,1), axis=0) / rho

    u[:, 0,:, 0] = u[0, :,:, 0] = u[Nx, :,:, 0] = u[:, Ny,:, 0]= u[:, :,0, 0]= 0.0
    u[:, :, Ny, 0] = U
    u[:, 0,:, 1] = u[0, :,:, 1] = u[Nx, :,:, 1] = u[:, Ny,:, 1]= u[:, :,0, 1]= u[:, :, Ny, 1] = 0.0
    u[:, 0,:, 2] = u[0, :,:, 2] = u[Nx, :,:, 2] = u[:, Ny,:, 2]= u[:, :,0, 2]= u[:, :, Ny, 2] = 0.0

    rho[:,:, 0] = (3.0*rho[:,:,1]-3.0*rho[:,:,2]+ 1.0*rho[:,:,3])
    rho[:,:, Nz] = (3.0*rho[:,:,Nz-1]-3.0*rho[:,:,Nz-2]+ 1.0*rho[:,:,Nz-3])
    rho[0, :,:] = (3.0*rho[1, :,:]- 3.0*rho[2, :,:]+ 1.0*rho[3, :,:])
    rho[Nx, :,:] = (3.0*rho[Nx-1,:,:]- 3.0*rho[Nx-2,:,:]+ 1.0*rho[Nx-3,:,:])
    rho[:, 0,:] = (3.0*rho[:,1,:]- 3.0*rho[:,2,:]+ 1.0*rho[:,3,:])
    rho[:, Ny,:] = (3.0*rho[:,Ny-1,:]- 3.0*rho[:,Ny-2,:]+ 1.0*rho[:,3,:])

    u[:, :,:, 0] = u[:, :,:, 0] + dt*(nu-nu_star)*u_d
    u[:, :,:, 1] = u[:, :,:, 1] + dt*(nu-nu_star)*v_d
    u[:, :,:, 2] = u[:, :,:, 2] + dt*(nu-nu_star)*w_d

    u[:, 0,:, 0] = u[0, :,:, 0] = u[Nx, :,:, 0] = u[:, Ny,:, 0]= u[:, :,0, 0]= 0.0
    u[:, :, Ny, 0] = U
    u[:, 0,:, 1] = u[0, :,:, 1] = u[Nx, :,:, 1] = u[:, Ny,:, 1]= u[:, :,0, 1]= u[:, :, Ny, 1] = 0.0
    u[:, 0,:, 2] = u[0, :,:, 2] = u[Nx, :,:, 2] = u[:, Ny,:, 2]= u[:, :,0, 2]= u[:, :, Ny, 2] = 0.0

    error = np.sum(np.sqrt((u_n-u[:, :, :, 0])**2+(v_n-u[:, :, :, 1])**2+(w_n-u[:, :, :, 2])**2))/np.sum(np.sqrt(u[:, :, :, 0]**2+u[:, :, :, 1]**2+u[:, :, :, 2]**2))
    u_n = u[:, :, :, 0].copy() 
    v_n = u[:, :, :, 1].copy()
    w_n = u[:, :, :, 2].copy()
    
    u_pd = possion_uu(Psi_qlbm1[t, :, :, :],dx)
    v_pd = possion_uu(Psi_qlbm2[t, :, :, :],dx)
    w_pd = possion_uu(Psi_qlbm3[t, :, :, :],dx)
    # Create and run the quantum circuit for LBM, rho
    qc = QLBMV2(density_field=Psi_qlbm[t, :, :, :], velocity_field=u_LBM, number_velocities=NUMBER_DISCRETE_VELOCITIES)
    
    # Compile and run the quantum circuit
    compiled_circuit = transpile(qc, simulator)
    result = simulator.run(compiled_circuit).result()
    
    # Extract and process the statevector
    statevector = np.array(result.get_statevector())
    real_part_statevector = np.real(statevector[:N_POINTS_X * N_POINTS_Y * N_POINTS_Z*27])
    reshaped_real_part = np.reshape(real_part_statevector, (N_POINTS_X, N_POINTS_Y, N_POINTS_Z,27),order='F')
    
    # Normalize and update Psi_qlbm
    norm_factor = np.linalg.norm(Psi_qlbm[t, :, :, :].flatten())
    fq = reshaped_real_part * norm_factor
    Psi_qlbm[t + 1,:,:, :] = np.sum(fq, axis=3)
    Psi_qlbm1[t + 1,:,:, :] = np.sum(fq * e[:,0].reshape(1,1,1,27), axis=3) / Psi_qlbm[t + 1,:,:, :]
    Psi_qlbm2[t + 1,:,:, :] = np.sum(fq * e[:,1].reshape(1,1,1,27), axis=3) / Psi_qlbm[t + 1,:,:, :]
    Psi_qlbm3[t + 1,:,:, :] = np.sum(fq * e[:,2].reshape(1,1,1,27), axis=3) / Psi_qlbm[t + 1,:,:, :]
    
    Psi_qlbm[t + 1,:,:, 0] = 3.0*Psi_qlbm[t + 1,:,:,1]-3.0*Psi_qlbm[t + 1,:,:,2] + 1.0*Psi_qlbm[t + 1,:,:,3]
    Psi_qlbm[t + 1,:,:, Nz] = 3.0*Psi_qlbm[t + 1,:,:,Nz-1]-3.0*Psi_qlbm[t + 1,:,:,Nz-2] + 1.0*Psi_qlbm[t + 1,:,:,Nz-3]
    Psi_qlbm[t + 1,0, :,:] = 3.0*Psi_qlbm[t + 1,1, :,:] - 3.0*Psi_qlbm[t + 1,2, :,:] + 1.0*Psi_qlbm[t + 1,3, :,:]
    Psi_qlbm[t + 1,Nx, :,:] = 3.0*Psi_qlbm[t + 1,Nx-1,:,:] - 3.0*Psi_qlbm[t + 1,Nx-2,:,:] + 1.0*Psi_qlbm[t + 1,Nx-3,:,:]
    Psi_qlbm[t + 1,:, 0,:] = 3.0*Psi_qlbm[t + 1,:,1,:] - 3.0*Psi_qlbm[t + 1,:,2,:] + 1.0*Psi_qlbm[t + 1,:,3,:]
    Psi_qlbm[t + 1,:, Ny,:] = 3.0*Psi_qlbm[t + 1,:,Ny-1,:] - 3.0*Psi_qlbm[t + 1,:,Ny-2,:] + 1.0*Psi_qlbm[t + 1,:,Ny-3,:] 
    
    # Create and run the quantum circuit for LBM, u
    Psi_qlbm1[t + 1,:, 0,:] = Psi_qlbm1[t + 1,0, :,:] = Psi_qlbm1[t + 1,Nx, :,:] = Psi_qlbm1[t + 1,:, Ny,:]= Psi_qlbm1[t + 1,:, :,0]= 0.0
    Psi_qlbm1[t + 1,:, :, Nz] = U
    

    # Create and run the quantum circuit for LBM, u
    Psi_qlbm2[t + 1,:, 0,:] = Psi_qlbm2[t + 1,0, :,:] = Psi_qlbm2[t + 1,Nx, :,:] = Psi_qlbm2[t + 1,:, Ny,:]= Psi_qlbm2[t + 1,:, :,0]= Psi_qlbm2[t + 1,:, :, Nz] = 0.0
    Psi_qlbm3[t + 1,:, 0,:] = Psi_qlbm3[t + 1,0, :,:] = Psi_qlbm3[t + 1,Nx, :,:] = Psi_qlbm3[t + 1,:, Ny,:]= Psi_qlbm3[t + 1,:, :,0]= Psi_qlbm3[t + 1,:, :, Nz] = 0.0

    Psi_qlbm1[t + 1,:, :,:] = Psi_qlbm1[t + 1,:, :,:] + dt*(nu-nu_star)*u_pd
    Psi_qlbm2[t + 1,:, :,:] = Psi_qlbm2[t + 1,:, :,:] + dt*(nu-nu_star)*v_pd
    Psi_qlbm3[t + 1,:, :,:] = Psi_qlbm3[t + 1,:, :,:] + dt*(nu-nu_star)*w_pd

    Psi_qlbm1[t + 1,:, 0,:] = Psi_qlbm1[t + 1,0, :,:] = Psi_qlbm1[t + 1,Nx, :,:] = Psi_qlbm1[t + 1,:, Ny,:]= Psi_qlbm1[t + 1,:, :,0]= 0.0
    Psi_qlbm1[t + 1,:, :, Nz] = U
    Psi_qlbm2[t + 1,:, 0,:] = Psi_qlbm2[t + 1,0, :,:] = Psi_qlbm2[t + 1,Nx, :,:] = Psi_qlbm2[t + 1,:, Ny,:]= Psi_qlbm2[t + 1,:, :,0]= Psi_qlbm2[t + 1,:, :, Nz] = 0.0
    Psi_qlbm3[t + 1,:, 0,:] = Psi_qlbm3[t + 1,0, :,:] = Psi_qlbm3[t + 1,Nx, :,:] = Psi_qlbm3[t + 1,:, Ny,:]= Psi_qlbm3[t + 1,:, :,0]= Psi_qlbm3[t + 1,:, :, Nz] = 0.0

    Psi_qlbm0 = Psi_qlbm[t + 1, :, :, :].copy()
    
    u_LBM[:,:, :,0] = Psi_qlbm1[t + 1, :, :, :]
    u_LBM[:,:, :,1] = Psi_qlbm2[t + 1, :, :, :]
    u_LBM[:,:, :,2] = Psi_qlbm3[t + 1, :, :, :]
    error1 = np.sum(np.sqrt((Psi_qlbm1[t + 1, :, :, :]-Psi_qlbm1[t, :, :, :])**2+(Psi_qlbm2[t + 1, :, :, :]-Psi_qlbm2[t, :, :, :])**2+(Psi_qlbm3[t + 1, :, :, :]-Psi_qlbm3[t, :, :, :])**2))/np.sum(np.sqrt(Psi_qlbm1[t + 1, :, :, :]**2+Psi_qlbm2[t + 1, :, :, :]**2+Psi_qlbm3[t + 1, :, :, :]**2))
    error2 = np.sum(np.sqrt((Psi_qlbm1[t + 1, :, :, :]-u_n)**2+(Psi_qlbm2[t + 1, :, :, :]-v_n)**2+(Psi_qlbm3[t + 1, :, :, :]-w_n)**2))/np.sum(np.sqrt(u_n**2+v_n**2+w_n**2))
    if t % 10 ==0:
        print(t+1,error,error1,error2)

    q_error.append(error1)
    c_error.append(error)
    qc_error.append(error2)

    if error1 < 1e-6 and error < 1e-6:
        break

1 1.0 0.027355623100385643 8.759422505247494e-14
11 0.020770355930923273 0.020770355930984946 5.235625921777534e-13
21 0.02022022169578874 0.020220221695819367 8.993636548268995e-13
31 0.02000091107044107 0.02000091107044948 1.1907173565104368e-12
41 0.017591801581510566 0.017591801581515895 1.2847788706461175e-12
51 0.0144179912672823 0.014417991267281056 1.2370186509823995e-12
61 0.009911198762885374 0.009911198762879488 1.0890664174394208e-12
71 0.009293312432355164 0.009293312432347704 8.592310356794144e-13
81 0.006798211904079986 0.006798211904080023 6.029005038540574e-13
91 0.006070342418978348 0.006070342418988436 3.893347163659295e-13
101 0.007871446721996028 0.00787144672200282 2.7893955515231655e-13
111 0.008544888763306899 0.008544888763306042 2.639666514916782e-13
121 0.006697239551048077 0.00669723955104196 3.212800919014881e-13
131 0.0030603438740593137 0.003060343874058133 4.826999879751417e-13
141 0.004784235189824594 0.004784235189833279 6.951501984053301e-13
151 0.006

In [ ]:
print(t+1,error,error1,error2)

In [ ]:
np.savetxt("Re=100_32/QLBM_u_"+str(t)+".csv",Psi_qlbm1[t+1,:,:,:].reshape(N_POINTS_X*N_POINTS_Y*N_POINTS_Z,1),delimiter=",")
np.savetxt("Re=100_32/QLBM_v_"+str(t)+".csv",Psi_qlbm2[t+1,:,:,:].reshape(N_POINTS_X*N_POINTS_Y*N_POINTS_Z,1),delimiter=",")
np.savetxt("Re=100_32/QLBM_w_"+str(t)+".csv",Psi_qlbm3[t+1,:,:,:].reshape(N_POINTS_X*N_POINTS_Y*N_POINTS_Z,1),delimiter=",")
np.savetxt("Re=100_32/QLBM_rho_"+str(t)+".csv",Psi_qlbm[t+1,:,:,:].reshape(N_POINTS_X*N_POINTS_Y*N_POINTS_Z,1),delimiter=",")
np.savetxt("Re=100_32/CLBM_u_"+str(t)+".csv",u[:,:,:,0].reshape(N_POINTS_X*N_POINTS_Y*N_POINTS_Z,1),delimiter=",")
np.savetxt("Re=100_32/CLBM_v_"+str(t)+".csv",u[:,:,:,1].reshape(N_POINTS_X*N_POINTS_Y*N_POINTS_Z,1),delimiter=",")
np.savetxt("Re=100_32/CLBM_w_"+str(t)+".csv",u[:,:,:,2].reshape(N_POINTS_X*N_POINTS_Y*N_POINTS_Z,1),delimiter=",")
np.savetxt("Re=100_32/CLBM_rho_"+str(t)+".csv",rho.reshape(N_POINTS_X*N_POINTS_Y*N_POINTS_Z,1),delimiter=",")
np.savetxt("Re=100_32/QLBM_error.csv",q_error,delimiter=",")
np.savetxt("Re=100_32/CLBM_error.csv",c_error,delimiter=",")
np.savetxt("Re=100_32/QLBM_CLBM_error.csv",qc_error,delimiter=",")